In [ ]:
print("Financial Agent Environment Ready!")

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

In [ ]:
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Who is the president of the USA?"}]
)

print(response.choices[0].message.content)

In [ ]:
with open("data/nexus_financial_annual_report_2023.txt", "r", encoding="utf-8") as f:
    text_content = f.read()

In [ ]:
print("🔍 FIRST 500 CHARACTERS:")
print(text_content[:500])

In [ ]:
from pypdf import PdfReader

reader = PdfReader("data/Amex_2021.pdf") 

print("📊 AMEX PDF STATS:")
print(f"Total Pages: {len(reader.pages)}")
print("-" * 50)

# Extract and print text from the very first page (Index 0)
first_page = reader.pages[3]
pdf_text = first_page.extract_text()

print("🔍 FIRST PAGE TEXT:")
print(pdf_text[:1000])  # Print first 1000 characters
print("-" * 50)

In [ ]:
## Reading all files together

In [ ]:
import os
from pypdf import PdfReader

print("🚀 Starting Master Data Ingestion...")
all_chunks = []

# Define chunk size parameters (Roughly 1000 characters with 200 character overlap)
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Let's list the specific files we want to process
files_to_process = [
    "Amex_2021.pdf", "Amex_2022.pdf", "Amex_2023.pdf", 
    "Amex_2024.pdf", "Amex_2025.pdf", "nexus_financial_annual_report_2023.txt"
]

for filename in files_to_process:
    filepath = os.path.join("data", filename)
    
    # Check if the file actually exists in the folder
    if not os.path.exists(filepath):
        print(f"⚠️ Warning: Could not find {filename} in 'data' folder. Skipping.")
        continue
        
    print(f"📖 Processing: {filename}...")
    
    # CASE 1: Handle PDF Files
    if filename.endswith(".pdf"):
        reader = PdfReader(filepath)
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if text:
                # Basic helper to slice text into chunks manually
                start = 0
                while start < len(text):
                    end = start + CHUNK_SIZE
                    chunk = text[start:end]
                    all_chunks.append({
                        "text": chunk,
                        "source": filename,
                        "page": page_num + 1
                    })
                    start += (CHUNK_SIZE - CHUNK_OVERLAP)

    # CASE 2: Handle the Text File
    elif filename.endswith(".txt"):
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
            start = 0
            while start < len(text):
                end = start + CHUNK_SIZE
                chunk = text[start:end]
                all_chunks.append({
                    "text": chunk,
                    "source": filename,
                    "page": "N/A"
                })
                start += (CHUNK_SIZE - CHUNK_OVERLAP)

print("\n✅ Extraction Complete!")
print(f"📊 Total document chunks created for the brain: {len(all_chunks)}")
if all_chunks:
    print(f"💡 Sample chunk format from {all_chunks[0]['source']}:")
    print(f"--- TEXT ---\n{all_chunks[0]['text'][:300]}...\n------------")

In [ ]:
type(all_chunks)

In [32]:
len(all_chunks)

4445

In [ ]:
all_chunks[1]

In [ ]:
## Embedding and Vector Database

In [ ]:
# import chromadb
# from chromadb.utils import embedding_functions

In [ ]:
# openai_ef = embedding_functions.OpenAIEmbeddingFunction(
#     api_key=os.environ.get("OPENAI_API_KEY"),
#     model_name="text-embedding-3-small"
# )

# chroma_client = chromadb.PersistentClient(path="chroma_storage")
# collection_name = "amex_financial_data"
# try:
#     chroma_client.delete_collection(name=collection_name)
# except Exception:
#     pass  # It didn't exist yet, which is perfectly fine

# collection = chroma_client.create_collection(
#     name=collection_name,
#     embedding_function=openai_ef
# )

In [31]:
import os
import chromadb
from openai import OpenAI

print("🤖 Step 1: Initializing OpenAI client manually...")
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print("⚙️ Step 2: Connecting to standard Chroma...")
chroma_client = chromadb.PersistentClient(path="chroma_storage")

collection_name = "amex_financial_data"
try:
    chroma_client.delete_collection(name=collection_name)
except Exception:
    pass  

# We create the collection WITHOUT an embedding function. 
# This tells Chroma: "Don't touch the internet, I will handle the math!"
collection = chroma_client.create_collection(name=collection_name)

print("\n🧠 Step 3: Generating embeddings via OpenAI & feeding Chroma in batches...")
# Processing in batches of 100 to keep API calls clean and fast
BATCH_SIZE = 100
total_chunks = len(all_chunks)

for i in range(0, total_chunks, BATCH_SIZE):
    batch = all_chunks[i:i + BATCH_SIZE]
    
    batch_texts = [chunk["text"] for chunk in batch]
    batch_metadatas = [{"source": chunk["source"], "page": str(chunk["page"])} for chunk in batch]
    batch_ids = [f"doc_chunk_{j}" for j in range(i, i + len(batch))]
    
    # 1. Call OpenAI directly for this specific batch's vectors
    response = openai_client.embeddings.create(
        input=batch_texts,
        model="text-embedding-3-small"
    )
    batch_embeddings = [emb.embedding for emb in response.data]
    
    # 2. Feed the pre-calculated numbers straight into our local database cabinet
    collection.add(
        documents=batch_texts,
        embeddings=batch_embeddings,
        metadatas=batch_metadatas,
        ids=batch_ids
    )
    
    print(f"  🔹 Progress: Synced chunks {i} to {i + len(batch)} / {total_chunks}")

print("-" * 50)
print(f"📊 FINAL VERIFICATION: Total chunks safely stored: {collection.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


🤖 Step 1: Initializing OpenAI client manually...
⚙️ Step 2: Connecting to standard Chroma...

🧠 Step 3: Generating embeddings via OpenAI & feeding Chroma in batches...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  🔹 Progress: Synced chunks 0 to 100 / 4445
  🔹 Progress: Synced chunks 100 to 200 / 4445
  🔹 Progress: Synced chunks 200 to 300 / 4445
  🔹 Progress: Synced chunks 300 to 400 / 4445
  🔹 Progress: Synced chunks 400 to 500 / 4445
  🔹 Progress: Synced chunks 500 to 600 / 4445
  🔹 Progress: Synced chunks 600 to 700 / 4445
  🔹 Progress: Synced chunks 700 to 800 / 4445
  🔹 Progress: Synced chunks 800 to 900 / 4445
  🔹 Progress: Synced chunks 900 to 1000 / 4445
  🔹 Progress: Synced chunks 1000 to 1100 / 4445
  🔹 Progress: Synced chunks 1100 to 1200 / 4445
  🔹 Progress: Synced chunks 1200 to 1300 / 4445
  🔹 Progress: Synced chunks 1300 to 1400 / 4445
  🔹 Progress: Synced chunks 1400 to 1500 / 4445
  🔹 Progress: Synced chunks 1500 to 1600 / 4445
  🔹 Progress: Synced chunks 1600 to 1700 / 4445
  🔹 Progress: Synced chunks 1700 to 1800 / 4445
  🔹 Progress: Synced chunks 1800 to 1900 / 4445
  🔹 Progress: Synced chunks 1900 to 2000 / 4445
  🔹 Progress: Synced chunks 2000 to 2100 / 4445
  🔹 Progress:

In [33]:
# Peek inside the collection to fetch the first 3 stored rows
peek_results = collection.get(
    limit=3,
    include=["documents", "metadatas", "embeddings"]
)

# Loop through and inspect what was pulled
for i in range(3):
    print(f"🆔 ID: {peek_results['ids'][i]}")
    print(f"📂 Metadata: {peek_results['metadatas'][i]}")
    print(f"📝 Raw Text Snippet: {peek_results['documents'][i][:150]}...")
    
    # Show just the first 5 dimensions of the 1,536 total numbers
    sample_vector = peek_results['embeddings'][i][:5]
    print(f"🔢 Vector Embedding (First 5 of 1536 dimensions):\n {sample_vector}")
    print("-" * 60)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


🆔 ID: doc_chunk_0
📂 Metadata: {'page': '1', 'source': 'Amex_2021.pdf'}
📝 Raw Text Snippet: 2021 ANNUAL REPORT...
🔢 Vector Embedding (First 5 of 1536 dimensions):
 [0.0033111572265625, 0.0117950439453125, 0.0258026123046875, 0.009552001953125, 0.01904296875]
------------------------------------------------------------
🆔 ID: doc_chunk_1
📂 Metadata: {'page': '3', 'source': 'Amex_2021.pdf'}
📝 Raw Text Snippet:  
i 
 
DEAR SHAREHOLDERS:  
I am exceptionally proud of our 2021 performance, which was one of the best in the company’s history thanks to the 
unwave...
🔢 Vector Embedding (First 5 of 1536 dimensions):
 [0.05352783203125, -0.0276641845703125, 0.038787841796875, 0.040740966796875, 0.053985595703125]
------------------------------------------------------------
🆔 ID: doc_chunk_10
📂 Metadata: {'page': '4', 'source': 'Amex_2021.pdf'}
📝 Raw Text Snippet: uarters with additional locations set to welcome back 
colleagues over the next several months.  
As we think about the future of work, 

In [34]:
def query_vector_db(search_query, n_results=3):
    """
    Takes a search string, converts it to a vector, 
    and returns the top N matching text paragraphs from Chroma.
    """
    # 1. Convert the search string into a vector manually
    response = openai_client.embeddings.create(
        input=[search_query],
        model="text-embedding-3-small"
    )
    query_vector = response.data[0].embedding
    
    # 2. Search Chroma using the raw vector numbers
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=n_results
    )
    
    # Return the documents and metadatas lists
    return results.get('documents', [[]])[0], results.get('metadatas', [[]])[0]

print("✅ Query Tool loaded and ready for testing!")

✅ Query Tool loaded and ready for testing!


In [35]:
# 1. Run our new tool
matched_texts, matched_metas = query_vector_db("What are the provisions for credit losses?")

# 2. Print out the very first match it found
print(f"📄 Top Match Source: {matched_metas[0]['source']} (Page {matched_metas[0]['page']})")
print(f"📝 Text Found:\n{matched_texts[0][:400]}...")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


📄 Top Match Source: Amex_2025.pdf (Page 130)
📝 Text Found:
cores where available, delinquency status, tenure of balance outstanding, amongst others. Credit losses on accrued 
interest are measured and presented as part of Reserves for credit losses on the Consolidated Balance Sheets and within the 
Provisions for credit losses in the Consolidated Statements of Income, rather than reversing interest income. 
For Other loans, we use vintage-based historical...


In [36]:
import os
import sys
from contextlib import redirect_stderr

def query_vector_db(search_query, n_results=8):
    """
    Converts search query to a vector and returns the top N matching text paragraphs.
    Mutes all internal library telemetry warnings completely.
    """
    # 1. Convert search string to vector
    response = openai_client.embeddings.create(
        input=[search_query],
        model="text-embedding-3-small"
    )
    query_vector = response.data[0].embedding
    
    # 2. Query Chroma while suppressing stderr output (silences the telemetry warning)
    with open(os.devnull, 'w') as fnull:
        with redirect_stderr(fnull):
            results = collection.query(
                query_embeddings=[query_vector],
                n_results=n_results
            )
            
    return results.get('documents', [[]])[0], results.get('metadatas', [[]])[0]

print("🤫 Silent, High-Context Query Tool loaded successfully!")

🤫 Silent, High-Context Query Tool loaded successfully!


In [37]:
# 1. Run our silent, higher-context tool
texts, metas = query_vector_db("What are the provisions for credit losses?")

print(f"📊 Successfully retrieved {len(texts)} chunks from the database.\n")

# 2. Print out the sources and snippets to verify they are clean
for i in range(min(3, len(texts))):
    print(f"📌 MATCH #{i+1} | 📂 {metas[i]['source']} (Page {metas[i]['page']})")
    print(f"📝 {texts[i][:250]}...")
    print("-" * 50)

📊 Successfully retrieved 8 chunks from the database.

📌 MATCH #1 | 📂 Amex_2025.pdf (Page 130)
📝 cores where available, delinquency status, tenure of balance outstanding, amongst others. Credit losses on accrued 
interest are measured and presented as part of Reserves for credit losses on the Consolidated Balance Sheets and within the 
Provision...
--------------------------------------------------
📌 MATCH #2 | 📂 Amex_2025.pdf (Page 64)
📝  
46 
TABLE 3: PROVISIONS FOR CREDIT LOSSES SUMMARY 
Years Ended December 31,        Change  Change 
(Millions, except percentages)  2025  2024  2023  2025 vs. 2024  2024 vs. 2023 
Card Member loans               
Net write-offs  $ 3,868   $ 3,515   ...
--------------------------------------------------
📌 MATCH #3 | 📂 Amex_2024.pdf (Page 61)
📝 47
TABLE 3: PROVISIONS FOR CREDIT LOSSES SUMMARY
Years Ended December 31, Change Change
(Millions, except percentages) 2024 2023 2022 2024 vs. 2023 2023 vs. 2022
Card Member loans
Net write-offs $ 3,515 $ 2,486 $

In [38]:
## Evaluation of results

In [39]:
def evaluate_completeness(question, context, answer):
    """
    Acts as an automated judge to score the completeness of an answer
    based strictly on the retrieved context. Returns a score between 0.0 and 1.0.
    """
    evaluation_prompt = f"""
    You are an unbiased quality assurance judge evaluating a financial RAG system.
    
    CRITERIA FOR COMPLETENESS:
    An answer is complete (1.0) if it fully addresses all facets of the user's question using the context, 
    leaving out no major trends, years, or metrics present in the text. 
    An answer is partial/incomplete (0.0 - 0.5) if it provides a vague summary, lacks specific numbers 
    mentioned in the context, or leaves the user with obvious logical gaps.

    INPUT DATA:
    - User Question: {question}
    - Retrieved Context: {context}
    - Generated Answer: {answer}

    TASK:
    1. Provide a 2-sentence critique explaining if the answer is complete or partial.
    2. Output a final numerical score labeled exactly as 'SCORE: X.X' (e.g., SCORE: 0.85).
    """

    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": evaluation_prompt}],
        temperature=0.0 # Strict, deterministic evaluation
    )
    
    return response.choices[0].message.content

print("📊 Automated Evaluation Framework loaded!")

📊 Automated Evaluation Framework loaded!


In [40]:
# 1. Define our test parameters
test_question = "What are the key drivers of credit loss provisions across the reported years?"

# 2. Fetch the text chunks using our high-context tool
docs, metas = query_vector_db(test_question, n_results=5)
combined_context = "\n".join(docs)

# 3. Simulate a "Partial/Incomplete" Answer manually
partial_answer = (
    "The primary drivers of credit loss provisions are delinquency status and vintage-based historical "
    "loss experiences mentioned in Amex 2025. The provisions are recorded in the Consolidated Statements of Income."
)

print("⚖️ Sending data to the LLM Judge for evaluation...\n")

# 4. Run the evaluation framework
judgment = evaluate_completeness(
    question=test_question, 
    context=combined_context, 
    answer=partial_answer
)

print("📝 --- JUDGE'S VERDICT ---")
print(judgment)

⚖️ Sending data to the LLM Judge for evaluation...

📝 --- JUDGE'S VERDICT ---
1. The generated answer is incomplete as it fails to address the specific key drivers of credit loss provisions across the reported years, such as net write-offs, reserve builds/releases, changes in macroeconomic outlook, and portfolio quality improvements. It also does not mention any specific numbers or trends from the context, leaving out critical details that are necessary for a comprehensive understanding of the credit loss provisions.

2. SCORE: 0.0


In [41]:
## Multi agent experimentation

In [42]:
print("🕵️‍♂️ 1. Researcher Agent is extracting raw financial facts...")
researcher_instruction = (
    "You are a meticulous Financial Researcher. Read the context and pull out EVERY single "
    "specific metric, year, net write-off number, reserve build/release, and macroeconomic trend. "
    "Output them as a highly detailed, unedited list of raw facts."
)

researcher_response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": researcher_instruction},
        {"role": "user", "content": f"CONTEXT:\n{combined_context}"}
    ],
    temperature=0.0
)
raw_facts = researcher_response.choices[0].message.content


print("✍️ 2. Writer Agent is synthesizing the final executive report...")
writer_instruction = (
    "You are an expert Wall Street Financial Analyst. Take the raw facts provided by the researcher "
    "and turn them into a comprehensive, highly complete executive summary. Ensure no year, metric, "
    "or trend is left out."
)

writer_response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": writer_instruction},
        {"role": "user", "content": f"RAW FACTS:\n{raw_facts}\n\nQUESTION: {test_question}"}
    ],
    temperature=0.2
)
agent_final_answer = writer_response.choices[0].message.content

print("\n🚀 Multi-Agent Answer Generated! Let's send it back to our judge...\n")

# 3. Re-evaluate our new agentic answer
new_judgment = evaluate_completeness(
    question=test_question, 
    context=combined_context, 
    answer=agent_final_answer
)

print("📝 --- NEW JUDGE'S VERDICT ---")
print(new_judgment)

🕵️‍♂️ 1. Researcher Agent is extracting raw financial facts...
✍️ 2. Writer Agent is synthesizing the final executive report...

🚀 Multi-Agent Answer Generated! Let's send it back to our judge...

📝 --- NEW JUDGE'S VERDICT ---
1. The answer is complete as it thoroughly addresses the key drivers of credit loss provisions across the reported years, providing specific numbers and trends for net write-offs, reserve builds/releases, and total provisions for Card Member loans, Card Member receivables, and Other loans and receivables. It also identifies the macroeconomic outlook and portfolio performance as significant influences, leaving no major gaps in the explanation.

2. SCORE: 1.0


In [43]:
print("🕵️‍♂️ --- WHAT THE RESEARCHER EXTRACTED ---")
print(raw_facts)
print("\n" + "="*60 + "\n")
print("✍️ --- WHAT THE WRITER ACTUALLY DELIVERED ---")
print(agent_final_answer)

🕵️‍♂️ --- WHAT THE RESEARCHER EXTRACTED ---
1. Net write-offs for Card Member loans in 2025: $3,868 million.
2. Net write-offs for Card Member loans in 2024: $3,515 million.
3. Net write-offs for Card Member loans in 2023: $2,486 million.
4. Change in net write-offs for Card Member loans from 2025 vs. 2024: $353 million, 10%.
5. Change in net write-offs for Card Member loans from 2024 vs. 2023: $1,029 million, 41%.
6. Reserve build (release) for Card Member loans in 2025: $198 million.
7. Reserve build (release) for Card Member loans in 2024: $594 million.
8. Reserve build (release) for Card Member loans in 2023: $1,353 million.
9. Change in reserve build (release) for Card Member loans from 2025 vs. 2024: $(396) million, (67%).
10. Change in reserve build (release) for Card Member loans from 2024 vs. 2023: $(759) million, (56%).
11. Total provisions for Card Member loans in 2025: $4,067 million.
12. Total provisions for Card Member loans in 2024: $4,109 million.
13. Total provisions f

In [44]:
### Pipeline

In [68]:
def run_multi_agent_pipeline(
    question, 
    context_chunks, 
    researcher_temp=0.0,  # Hyperparameter for Step 1
    writer_temp=0.0       # Hyperparameter for Step 2 (Deflated to 0.0 to squash hallucinations)
):
    """
    Automates the sequential handoff between agents using adjustable hyperparameters.
    """
    # Combine retrieved text chunks into one single text dump (y)
    full_context_dump = "\n\n".join(context_chunks)
    
    # 1. Trigger the Researcher Agent (Extraction Phase)
    researcher_instruction = (
        "You are a meticulous Financial Researcher. Read the context and pull out EVERY single "
        "specific metric, year, net write-off number, reserve build/release, and macroeconomic trend. "
        "Output them as a highly detailed, unedited list of raw facts. Pay strict attention to "
        "negative constraints like 'NOT', 'EXCEPT', or 'WITHOUT'."
    )
    
    researcher_response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": researcher_instruction},
            {"role": "user", "content": f"CONTEXT:\n{full_context_dump}"}
        ],
        temperature=researcher_temp # Parameterized
    )
    extracted_facts = researcher_response.choices[0].message.content

    # 2. Automatically pass the extracted facts to the Writer Agent (Synthesis Phase)
    writer_instruction = (
        "You are an expert Wall Street Financial Analyst. Take the raw facts provided by the researcher "
        "and turn them into a comprehensive, highly complete executive summary answering the question. "
        "Ensure no year, metric, or trend is left out. Stick strictly to the provided facts without extrapolation."
    )

    writer_response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": writer_instruction},
            {"role": "user", "content": f"RAW FACTS:\n{extracted_facts}\n\nQUESTION: {question}"}
        ],
        temperature=writer_temp # Parameterized
    )
    final_summary = writer_response.choices[0].message.content
    
    return final_summary

In [46]:
## Testing

In [47]:
# 1. Define a quick test question
single_test_question = "What happened to Card Member loan write-offs between 2023 and 2025?"

# 2. Query our vector database to get the raw chunks
retrieved_chunks, _ = query_vector_db(single_test_question, n_results=5)

print("⚡ Running the automated multi-agent pipeline function...")

# 3. Call our brand new function
test_prediction = run_multi_agent_pipeline(single_test_question, retrieved_chunks)

print("\n✍️ --- PIPELINE OUTPUT ---")
print(test_prediction)

⚡ Running the automated multi-agent pipeline function...

✍️ --- PIPELINE OUTPUT ---
**Executive Summary:**

The period from 2020 to 2025 has been marked by significant fluctuations in the reserve for credit losses and the associated provisions and write-offs for Card Member loans, reflecting changes in the macroeconomic environment and loan portfolio dynamics.

**2020 Overview:**
- The reserve for credit losses increased significantly due to the adverse global macroeconomic outlook caused by the COVID-19 pandemic. This was partially offset by a decline in outstanding loan balances and lower delinquencies.
- The beginning balance for the reserve was $4,027 million, with provisions amounting to $3,453 million.
- Net write-offs for principal and interest/fees were $(1,795) million and $(375) million, respectively.
- The ending balance for the year was $5,344 million.

**2021 Overview:**
- The reserve decreased as portfolio quality improved and macroeconomic forecasts became more favorabl

In [48]:
## Validation 

In [49]:
# Our Mini Golden Dataset / Validation Set
validation_questions = [
    "What are the key drivers of credit loss provisions across the reported years?",
    "How did the net write-offs for Card Member receivables change between 2023 and 2025?",
    "What caused the reserve build or release adjustments for Other loans in the current year?"
]

print(f"📋 Loaded Validation Dataset with N = {len(validation_questions)} sample rows.")

📋 Loaded Validation Dataset with N = 3 sample rows.


In [50]:
## batch evaluation

In [51]:
import re

# Arrays to hold our population metrics
all_scores = []
pipeline_results = []

print("🚀 Starting Batch Validation Pipeline...\n")

for i, question in enumerate(validation_questions, 1):
    print(f"📊 Processing Row {i}/{len(validation_questions)}: '{question}'")
    
    # 1. Fetch the actual raw ground truth chunks (y)
    retrieved_chunks, _ = query_vector_db(question, n_results=5)
    
    # 2. Run the multi-agent generation pipeline to get the prediction (ŷ)
    prediction = run_multi_agent_pipeline(question, retrieved_chunks)
    
    # 3. Audit using our isolated Judge function
    context_dump = "\n".join(retrieved_chunks)
    judge_verdict = evaluate_completeness(question, context_dump, prediction)
    
    # 4. Parse out the numerical score from the verdict text using regex
    # Looks for the pattern 'SCORE: X.X'
    score_match = re.search(r"SCORE:\s*([0-9.]+)", judge_verdict)
    score = float(score_match.group(1)) if score_match else 0.0
    
    # Store results
    all_scores.append(score)
    pipeline_results.append({
        "question": question,
        "prediction": prediction,
        "verdict": judge_verdict,
        "score": score
    })
    
    print(f"✅ Row {i} Completed. Score awarded: {score}\n")

print("🏁 Batch processing finished successfully!")

🚀 Starting Batch Validation Pipeline...

📊 Processing Row 1/3: 'What are the key drivers of credit loss provisions across the reported years?'
✅ Row 1 Completed. Score awarded: 1.0

📊 Processing Row 2/3: 'How did the net write-offs for Card Member receivables change between 2023 and 2025?'
✅ Row 2 Completed. Score awarded: 1.0

📊 Processing Row 3/3: 'What caused the reserve build or release adjustments for Other loans in the current year?'
✅ Row 3 Completed. Score awarded: 1.0

🏁 Batch processing finished successfully!


In [52]:
### SUspect overfitting lets do this

In [53]:
# Our New Stress-Test Validation Set
validation_questions = [
    # Clean baseline questions (The ones that got 1.0)
    "What are the key drivers of credit loss provisions across the reported years?",
    "How did the net write-offs for Card Member receivables change between 2023 and 2025?",
    
    # NEW: The Abstract Question (Harder semantic extraction)
    "What is the overall management sentiment regarding macroeconomic uncertainty and delinquency seasoning?",
    
    # NEW: The Out-of-Bounds Question (Testing for hallucination and edge-case failure)
    "What were the total net write-offs for cryptocurrency-backed auto loans in 2025?"
]

print(f"🕵️‍♂️ Expanded Validation Dataset to N = {len(validation_questions)} rows with adversarial edge cases.")

🕵️‍♂️ Expanded Validation Dataset to N = 4 rows with adversarial edge cases.


In [54]:
import re

# Arrays to hold our population metrics
all_scores = []
pipeline_results = []

print("🚀 Starting Batch Validation Pipeline...\n")

for i, question in enumerate(validation_questions, 1):
    print(f"📊 Processing Row {i}/{len(validation_questions)}: '{question}'")
    
    # 1. Fetch the actual raw ground truth chunks (y)
    retrieved_chunks, _ = query_vector_db(question, n_results=5)
    
    # 2. Run the multi-agent generation pipeline to get the prediction (ŷ)
    prediction = run_multi_agent_pipeline(question, retrieved_chunks)
    
    # 3. Audit using our isolated Judge function
    context_dump = "\n".join(retrieved_chunks)
    judge_verdict = evaluate_completeness(question, context_dump, prediction)
    
    # 4. Parse out the numerical score from the verdict text using regex
    # Looks for the pattern 'SCORE: X.X'
    score_match = re.search(r"SCORE:\s*([0-9.]+)", judge_verdict)
    score = float(score_match.group(1)) if score_match else 0.0
    
    # Store results
    all_scores.append(score)
    pipeline_results.append({
        "question": question,
        "prediction": prediction,
        "verdict": judge_verdict,
        "score": score
    })
    
    print(f"✅ Row {i} Completed. Score awarded: {score}\n")

print("🏁 Batch processing finished successfully!")

🚀 Starting Batch Validation Pipeline...

📊 Processing Row 1/4: 'What are the key drivers of credit loss provisions across the reported years?'
✅ Row 1 Completed. Score awarded: 1.0

📊 Processing Row 2/4: 'How did the net write-offs for Card Member receivables change between 2023 and 2025?'
✅ Row 2 Completed. Score awarded: 1.0

📊 Processing Row 3/4: 'What is the overall management sentiment regarding macroeconomic uncertainty and delinquency seasoning?'
✅ Row 3 Completed. Score awarded: 1.0

📊 Processing Row 4/4: 'What were the total net write-offs for cryptocurrency-backed auto loans in 2025?'
✅ Row 4 Completed. Score awarded: 0.0

🏁 Batch processing finished successfully!


In [55]:
## remove bias from validation question

In [59]:
import random

def generate_synthetic_test_set(chroma_collection, num_questions=10):
    """
    Randomly samples chunks from ChromaDB and uses an isolated LLM 
    to programmatically generate an ABSOLUTELY UNIQUE validation dataset.
    """
    print("🧬 Accessing ChromaDB storage...")
    
    # 1. Fetch all documents currently stored in your collection
    all_db_data = chroma_collection.get(include=["documents"])
    all_chunks = all_db_data["documents"]
    
    if not all_chunks:
        print("❌ Error: Your ChromaDB collection appears to be empty.")
        return []
        
    # 2. Use a Python Set to enforce strict uniqueness at the data level
    synthetic_bench = set()
    attempts = 0
    max_attempts = num_questions * 3  # Safety break to prevent an infinite loop
    
    print(f"🧠 Extracting unique factual questions from database chunks...\n")
    
    generator_instruction = (
        "You are an objective AI Data Annotator. Your task is to read the provided text chunk "
        "and write ONE explicit, highly targeted question that can be answered completely and "
        "unambiguously by reading this chunk.\n\n"
        "RULES:\n"
        "1. The question must focus on specific metrics, financial values, dates, or core drivers mentioned.\n"
        "2. Do not use outside knowledge. The answer must exist entirely inside the chunk.\n"
        "3. Output ONLY the question text. Do not include headers, numbering, or explanations."
    )
    
    # 3. Loop runs dynamically until the SET reaches our desired size
    while len(synthetic_bench) < num_questions and attempts < max_attempts:
        attempts += 1
        
        # Pick a random chunk from the pool
        chunk = random.choice(all_chunks)
        
        response = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": generator_instruction},
                {"role": "user", "content": f"TEXT CHUNK:\n{chunk}"}
            ],
            temperature=0.5
        )
        
        generated_question = response.choices[0].message.content.strip()
        
        # Check if it's a duplicate before printing/adding
        if generated_question not in synthetic_bench:
            synthetic_bench.add(generated_question)
            print(f"✨ Generated Unique Question {len(synthetic_bench)}: '{generated_question}'")
        else:
            print("⚠️ Skipped a duplicate or highly repetitive question. Retrying with a new chunk...")
        
    print("\n🏁 Absolute Unique Golden Dataset successfully created!")
    
    # Convert set back to a normal python list so our batch loop can read it by index
    return list(synthetic_bench)

In [60]:
# Programmatically create a robust 10-question validation set
automated_validation_questions = generate_synthetic_test_set(collection, num_questions=10)

🧬 Accessing ChromaDB storage...
🧠 Extracting unique factual questions from database chunks...

✨ Generated Unique Question 1: 'What level of the fair value hierarchy are the investment securities classified within, according to the pricing models used?'
✨ Generated Unique Question 2: 'What was the percentage of principal, interest, and fees net write-offs to average small business loans outstanding for the first value mentioned?'
✨ Generated Unique Question 3: 'What was the amount of net income reported for the year ending December 31, 2022?'
✨ Generated Unique Question 4: 'What is not dependent on the credit rating according to the text?'
✨ Generated Unique Question 5: 'What is the change in interest income for U.S. Card Member and Other loans?'
✨ Generated Unique Question 6: 'What is the target range for American Express Company's Common Equity Tier 1 (CET1) risk-based capital ratio?'
✨ Generated Unique Question 7: 'What factors were considered in the statistical and analytical model

In [61]:
## evaluation

In [62]:
import re
import numpy as np

# Arrays to hold our population metrics
all_scores = []
pipeline_results = []

print("🚀 Starting 10-Row Batch Validation Pipeline...\n")

for i, question in enumerate(automated_validation_questions, 1):
    print(f"📊 Processing Row {i}/10: '{question}'")
    
    # 1. Fetch actual raw ground truth chunks (y)
    retrieved_chunks, _ = query_vector_db(question, n_results=5)
    
    # 2. Run the multi-agent generation pipeline to get the prediction (ŷ)
    prediction = run_multi_agent_pipeline(question, retrieved_chunks)
    
    # 3. Audit using your isolated Judge function
    context_dump = "\n".join(retrieved_chunks)
    judge_verdict = evaluate_completeness(question, context_dump, prediction)
    
    # 4. Extract numerical score using the regex match object check we built
    score_match = re.search(r"SCORE:\s*([0-9.]+)", judge_verdict)
    score = float(score_match.group(1)) if score_match else 0.0
    
    # Store results
    all_scores.append(score)
    pipeline_results.append({
        "question": question,
        "prediction": prediction,
        "score": score
    })
    
    print(f"✅ Row {i} Completed. Score: {score}\n")

print("🏁 Batch processing finished successfully!\n")

# Compute final population statistics
mean_score = np.mean(all_scores)
std_dev = np.std(all_scores)

print("📊 --- AUTOMATED SYSTEM PERFORMANCE REPORT ---")
print(f"• Total Unbiased Sample Rows (N): {len(all_scores)}")
print(f"• Mean Completeness Score (μ): {mean_score:.2f}")
print(f"• Score Standard Deviation (σ): {std_dev:.2f}")
print("-" * 46)

if mean_score >= 0.90:
    print("🌟 Production Ready: The multi-agent architecture shows stable, comprehensive information recall.")
else:
    print("⚠️ Regression Detected: Variance or information loss is present. Prompt/chunk optimizations required.")

🚀 Starting 10-Row Batch Validation Pipeline...

📊 Processing Row 1/10: 'What factors were considered in the statistical and analytical models for reserves for credit losses under the incurred loss methodology as of December 31, 2019?'
✅ Row 1 Completed. Score: 1.0

📊 Processing Row 2/10: 'What was the percentage of principal, interest, and fees net write-offs to average small business loans outstanding for the first value mentioned?'
✅ Row 2 Completed. Score: 0.5

📊 Processing Row 3/10: 'What types of claims are involved in the legal proceedings mentioned in the text chunk?'
✅ Row 3 Completed. Score: 1.0

📊 Processing Row 4/10: 'What was the primary driver for the increase in marketing and business development expenses?'
✅ Row 4 Completed. Score: 1.0

📊 Processing Row 5/10: 'What was the network volume in billions outside the U.S. for the year 2020?'
✅ Row 5 Completed. Score: 1.0

📊 Processing Row 6/10: 'What was the amount of net income reported for the year ending December 31, 2022?'

In [63]:
def evaluate_faithfulness(question, context, prediction):
    """
    Isolated evaluation function to audit the pipeline's output for hallucination.
    Checks if all claims in the prediction are explicitly supported by the context.
    """
    judge_prompt = (
        "You are an expert AI Quality Assurance Auditor specializing in financial data integrity.\n"
        "Your task is to judge if the PREDICTION is 100% faithful and strictly grounded in the provided CONTEXT.\n\n"
        "CRITERIA:\n"
        "1. Every metric, date, claim, and statistic in the PREDICTION must be explicitly supported by the CONTEXT.\n"
        "2. If the PREDICTION introduces outside information, historical facts not present, or extrapolates numbers, it is a HALLUCINATION.\n"
        "3. If the prediction states that the information is missing/unavailable because it wasn't in the context, that is considered FAITHFUL (Score 1.0).\n\n"
        "OUTPUT FORMAT:\n"
        "Provide a brief 2-sentence rationale detailing your factual audit.\n"
        "On the final line, you must write exactly: 'SCORE: X.X' (where 1.0 means perfectly faithful/no hallucinations, and 0.0 means fabricated data is present)."
    )
    
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": judge_prompt},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nPREDICTION:\n{prediction}"}
        ],
        temperature=0.0 # Deterministic grounding check
    )
    return response.choices[0].message.content.strip()

In [66]:
import re
import numpy as np

completeness_scores = []
faithfulness_scores = []

print("🚀 Starting Dual-Metric Production Validation (N=10)...\n")

for i, question in enumerate(automated_validation_questions, 1):
    print(f"📊 Processing Row {i}/10: '{question}'")
    
    # 1. Fetch raw ground truth chunks
    retrieved_chunks, _ = query_vector_db(question, n_results=5)
    context_dump = "\n".join(retrieved_chunks)
    
    # 2. Execute multi-agent generation pipeline
    prediction = run_multi_agent_pipeline(question, retrieved_chunks)
    
    # 3. Run Completeness Audit
    comp_verdict = evaluate_completeness(question, context_dump, prediction)
    comp_match = re.search(r"SCORE:\s*([0-9.]+)", comp_verdict)
    comp_score = float(comp_match.group(1)) if comp_match else 0.0
    completeness_scores.append(comp_score)
    
    # 4. Run Faithfulness Audit
    faith_verdict = evaluate_faithfulness(question, context_dump, prediction)
    faith_match = re.search(r"SCORE:\s*([0-9.]+)", faith_verdict)
    # Fixed the typo here:
    faith_score = float(faith_match.group(1)) if faith_match else 0.0
    faithfulness_scores.append(faith_score)
    
    print(f"✅ Row {i} Finished | Completeness: {comp_score} | Faithfulness: {faith_score}")

print("\n🏁 Validation suite completed successfully!")

# Compute System Metrics
print("\n📊 ================== SYSTEM PERFORMANCE MATRIX ==================")
print(f"• Mean Completeness Score (μ_comp) : {np.mean(completeness_scores):.2f} | Std Dev (σ_comp): {np.std(completeness_scores):.2f}")
print(f"• Mean Faithfulness Score (μ_faith) : {np.mean(faithfulness_scores):.2f} | Std Dev (σ_faith): {np.std(faithfulness_scores):.2f}")
print("==================================================================")

🚀 Starting Dual-Metric Production Validation (N=10)...

📊 Processing Row 1/10: 'What factors were considered in the statistical and analytical models for reserves for credit losses under the incurred loss methodology as of December 31, 2019?'
✅ Row 1 Finished | Completeness: 1.0 | Faithfulness: 1.0
📊 Processing Row 2/10: 'What was the percentage of principal, interest, and fees net write-offs to average small business loans outstanding for the first value mentioned?'
✅ Row 2 Finished | Completeness: 1.0 | Faithfulness: 1.0
📊 Processing Row 3/10: 'What types of claims are involved in the legal proceedings mentioned in the text chunk?'
✅ Row 3 Finished | Completeness: 1.0 | Faithfulness: 1.0
📊 Processing Row 4/10: 'What was the primary driver for the increase in marketing and business development expenses?'
✅ Row 4 Finished | Completeness: 1.0 | Faithfulness: 1.0
📊 Processing Row 5/10: 'What was the network volume in billions outside the U.S. for the year 2020?'
✅ Row 5 Finished | Comple

In [67]:
## changing to n=8

In [69]:
import re
import numpy as np

completeness_scores = []
faithfulness_scores = []
pipeline_results = []

print("🚀 Starting Hyperparameter-Tuned Dual-Metric Pipeline (N=10)...\n")

for i, question in enumerate(automated_validation_questions, 1):
    print(f"📊 Processing Row {i}/10: '{question}'")
    
    # 1. TUNING RETRIEVAL: Expand window from 5 to 8 chunks to minimize information gaps
    retrieved_chunks, _ = query_vector_db(question, n_results=8)
    context_dump = "\n".join(retrieved_chunks)
    
    # 2. TUNING GENERATION: Injecting strict 0.0 deterministic temperatures to crush hallucinations
    prediction = run_multi_agent_pipeline(
        question, 
        retrieved_chunks, 
        researcher_temp=0.0, 
        writer_temp=0.0
    )
    
    # 3. Run Completeness Audit
    comp_verdict = evaluate_completeness(question, context_dump, prediction)
    comp_match = re.search(r"SCORE:\s*([0-9.]+)", comp_verdict)
    comp_score = float(comp_match.group(1)) if comp_match else 0.0
    completeness_scores.append(comp_score)
    
    # 4. Run Faithfulness Audit
    faith_verdict = evaluate_faithfulness(question, context_dump, prediction)
    faith_match = re.search(r"SCORE:\s*([0-9.]+)", faith_verdict)
    faith_score = float(faith_match.group(1)) if faith_match else 0.0
    faithfulness_scores.append(faith_score)
    
    pipeline_results.append({
        "question": question,
        "prediction": prediction,
        "completeness": comp_score,
        "faithfulness": faith_score
    })
    
    print(f"✅ Row {i} Finished | Completeness: {comp_score} | Faithfulness: {faith_score}\n")

print("🏁 Optimized batch processing finished successfully!\n")

# Compute final tuned population statistics
mean_comp = np.mean(completeness_scores)
std_comp = np.std(completeness_scores)
mean_faith = np.mean(faithfulness_scores)
std_faith = np.std(faithfulness_scores)

print("📊 ================== TUNED SYSTEM PERFORMANCE MATRIX ==================")
print(f"• Mean Completeness Score (μ_comp) : {mean_comp:.2f} | Std Dev (σ_comp): {std_comp:.2f}")
print(f"• Mean Faithfulness Score (μ_faith) : {mean_faith:.2f} | Std Dev (σ_faith): {std_faith:.2f}")
print("========================================================================")

if mean_comp >= 0.90 and mean_faith >= 0.95:
    print("🌟 Production Ready: Variance is stabilized and hallucinations are mitigated.")
else:
    print("⚠️ Further Tuning Required: Structural data gaps or synthesis anomalies still persist.")

🚀 Starting Hyperparameter-Tuned Dual-Metric Pipeline (N=10)...

📊 Processing Row 1/10: 'What factors were considered in the statistical and analytical models for reserves for credit losses under the incurred loss methodology as of December 31, 2019?'
✅ Row 1 Finished | Completeness: 1.0 | Faithfulness: 1.0

📊 Processing Row 2/10: 'What was the percentage of principal, interest, and fees net write-offs to average small business loans outstanding for the first value mentioned?'
✅ Row 2 Finished | Completeness: 1.0 | Faithfulness: 1.0

📊 Processing Row 3/10: 'What types of claims are involved in the legal proceedings mentioned in the text chunk?'
✅ Row 3 Finished | Completeness: 1.0 | Faithfulness: 1.0

📊 Processing Row 4/10: 'What was the primary driver for the increase in marketing and business development expenses?'
✅ Row 4 Finished | Completeness: 0.5 | Faithfulness: 1.0

📊 Processing Row 5/10: 'What was the network volume in billions outside the U.S. for the year 2020?'
✅ Row 5 Finis